In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
! git pull

## Add import path

In [ ]:
import os
import sys
import gc

In [ ]:
from pathlib import Path

In [ ]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')

In [ ]:
add_library_level(level=5)

In [ ]:
parent_dir1 = str(Path().resolve().parents[1])
# parent_dir2 = str(Path().resolve().parents[2])

# Insert the parent directory path at the beginning of sys.path
sys.path.insert(0, parent_dir1)
# sys.path.insert(0, parent_dir2)

# Now, you can import modules from the parent directory
# For example, if you have a file named 'my_module.py' in the parent directory
# from my_module import some_function

## Organize imports

In [ ]:
import math
from pathlib import Path

In [ ]:
from tqdm import tqdm

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, isspmatrix_csr, load_npz, save_npz

In [ ]:
import joblib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch

In [ ]:
from fuzzywuzzy import fuzz

In [ ]:
from src.lattmc.models.detokenizer import GPT2Detokenizer

## Initialize Paths

In [ ]:
PATH = Path('transcoders')
TR_PATH = PATH / 'data' / 'transcoders'
GPT2 = TR_PATH / 'gpt2'
OWT_TOKENS_DIR = GPT2 / 'owt_tokens'
DATASET_PATH = OWT_TOKENS_DIR / 'dataset.csv'
OWT_TEXT_PATH = OWT_TOKENS_DIR / 'text.joblib'
TOKENS_PATH = OWT_TOKENS_DIR / 'owt_tokens_torch.pt'
TR_EMBS_DIR = GPT2
SAE_EMBS_DIR = PATH.parent.parent.parent / 'sae' / 'data' / 'sae' / 'gpt2'
OWT_TOKENS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
! ls {TOKENS_PATH.absolute()}

In [ ]:
! ls {GPT2}

In [ ]:
! ls {PATH.parent.parent.parent / 'sae' / 'data' / 'sae' / 'gpt2'}

In [ ]:
! ls {DATASET_PATH}

## Initialize Tokenizers

In [ ]:
detok = GPT2Detokenizer()
text = detok.decode([15496, 11, 428, 318, 257, 1332, 13])
# "Hello, this is a test."
print(text)

## Analyze Text

In [ ]:
pd

In [ ]:
df = pd.read_csv(DATASET_PATH)

In [ ]:
df

In [ ]:
df.shape

## Load Tokens

In [ ]:
tokens = torch.load(TOKENS_PATH, weights_only=False, map_location='cpu')

In [ ]:
tokens.shape

In [ ]:
with tqdm(tokens) as ptkens:
    texts = [detok.decode(
        tks, 
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ) for tks in ptkens]

In [ ]:
len(texts), len(texts[15].split())

In [ ]:
texts[20]

In [ ]:
df.iloc[21, 1]

In [ ]:
gc.collect()

## Match Texts

In [ ]:
max_part = 0
min_part = 100
max_full = 0
min_full = 100
dataset = dict(id=list(), text=list())
with tqdm(list(range(len(texts)))) as prange:
    for i in prange:
        text1 = texts[i].replace('<|endoftext|>', '')
        text2 = df.iloc[i + 1, 1]
        dataset['id'].append(i)
        dataset['text'].append(text2)
        partial_sim = fuzz.partial_ratio(text1, text2)
        full_sim = fuzz.ratio(text1, text2)
        max_part = partial_sim if max_part < partial_sim else max_part
        min_part = partial_sim if partial_sim < min_part else min_part
        max_full = full_sim if max_full < full_sim else max_full
        min_full = full_sim if full_sim < min_full else min_full
        if full_sim < 99:
            print(f'Min similarity for {i} is {min_full}%')
print(f'Max partial {max_part}%')
print(f'Min partial {min_part}%')
print(f'Max full {max_full}%')
print(f'Min full {min_full}%')

In [ ]:
idx = 232

In [ ]:
texts[idx].replace('<|endoftext|>', '')

In [ ]:
dataset['id'][idx], dataset['text'][idx]

In [ ]:
df.iloc[idx + 1, 1]

In [ ]:
full_sim = fuzz.ratio(texts[idx], df.iloc[idx + 1, 1])
full_sim

In [ ]:
df_dataset = pd.DataFrame(dataset, columns=['id', 'text'])
df_dataset

In [ ]:
df_dataset.to_csv(OWT_TOKENS_DIR / 'text.csv', index=False)

In [ ]:
df_dataset.shape

## Serialize Decoded text

In [ ]:
text_pt = None

In [ ]:
joblib.dump(texts, OWT_TEXT_PATH, compress=3)

In [ ]:
print(OWT_TEXT_PATH.absolute())

In [ ]:
text_pt = joblib.load(OWT_TEXT_PATH)

In [ ]:
len(text_pt)

## Compare Encoded and Encoded Back

In [ ]:
if text_pt is None:
    text_pt = texts

In [ ]:
# tokens_pt = detok.encode_concatenated(text_pt, return_tensors='pt')

In [ ]:
with tqdm(text_pt) as ptext:
        tokens_list = [
            detok.encode(
                f'{txt}', 
                add_bos_token=True,   # Add BOS like tokenize_and_concatenate does
                # max_length=128,       # Target length
                # padding=True,         # Pad shorter sequences
                # truncation=True,       # Truncate longer sequences
                return_tensors='np',
            )
            # Always returns exactly 128 tokens
            for txt in ptext
        ]

In [ ]:
tokens_pt = torch.tensor(tokens_list)

In [ ]:
detok.tokenizer.eos_token

In [ ]:
tokens_pt.shape

In [ ]:
neqs = torch.where(tokens != tokens_pt)

In [ ]:
neqs[0].shape

In [ ]:
torch.tensor(tokens_pt)

In [ ]:
tok_lens = dict()
for tok in tokens_list:
    tok_len = len(tok)
    tok_lens.setdefault(tok_len, 0)
    tok_lens[tok_len] += 1

In [ ]:
tok_lens

In [ ]:
tok_pt = detok.encode(text_pt[120], return_tensors='pt')
tok_pt.shape

In [ ]:
count_tks = 0
for tok in tokens_pt:
    tok_len = len(tok)
    if tok_len != 128:
        count_tks += 1
print(count_tks)

## Compare Embeddings

In [ ]:
comps = list()
with tqdm(list(range(12))) as prange:
    for i in prange:
        TV = load_npz(TR_EMBS_DIR / f'V{i}.npz')
        EV = load_npz(SAE_EMBS_DIR / f'V{i}.npz')
        TV = TV.toarray()
        EV = EV.toarray()
        # with np.load(TR_EMBS_DIR / f'V{i}.npz') as tdata:
        #     TV = tdata
        # with np.load(SAE_EMBS_DIR / f'V{i}.npz') as edata:
        #     EV = edata
        # print(f'{TV.shape = }, {EV.shape = }')
        comps.append((TV == EV))

In [ ]:
# eqs = np.where(comps[11])[0]

In [ ]:
# np.all(EV[eqs] == 0)